# Notebook 05 — Reliability and consensus aggregation

**Project:** Study 1 — Chatbot Agreeableness (DGK Lab)  
**Pipeline stage:** 5 of 8 (NLP Colab notebooks) + 1 integration Rmd  
**Runtime:** ~2 minutes (all local computation, no API calls)  
**API costs:** $0

## Purpose

Convert the triple-coded OE output from Notebook 04 into a manuscript-usable dataset. Three jobs:

1. **Reliability assessment.** For each (oe_field × code_id) combination, compute three inter-rater reliability statistics across the three LLMs, with bootstrap 95% CIs on Krippendorff's α.
2. **Tier assignment.** Classify each code as **confirmatory** (κ ≥ .60 on all three statistics), **exploratory** (.40–.60), or **descriptive only** (< .40). Conjunctive rule.
3. **Consensus aggregation.** Majority vote (≥ 2 of 3 LLMs) per (sona_id × code_id), with conservative tie-break (2 of 3 with 1 missing → tied → 0).

## Why three reliability statistics

Cross-family LLM coding is methodologically novel; we report Cohen's κ (pairwise mean), Fleiss's κ (omnibus), and Krippendorff's α (handles missingness) without designating a primary statistic. **Tier assignment is conjunctive**: a code reaches a tier only if all three statistics clear that tier's threshold. This is the most defensible rule for reviewers who care about any one of the three.

## Inputs

- `data/derived/coding/oe_coded_long.parquet` (from Notebook 04)
- `data/derived/coding/oe_coded_per_llm/*.parquet` (from Notebook 04, used only for sanity checks)
- `data/derived/rubric/rubric_v2.json` (from Notebook 03)

## Outputs

- `data/derived/coding/reliability_table.csv` — one row per (oe_field × code_id) with all reliability statistics, prevalence, sample size, and tier assignment
- `data/derived/coding/consensus_long.parquet` — long-form consensus values (one row per sona × oe × code)
- `data/derived/coding/consensus_wide_<oe_field>.parquet` — one wide-form parquet per OE question (rows = sona_id, columns = code_ids for that question). Direct merge with `analytic_data.csv` on sona_id.
- `data/derived/coding/reliability_per_condition.csv` — α-HighA and α-LowA per code, for diffusion check
- `data/derived/coding/code_prevalence_by_condition.csv` — descriptive prevalence by condition (no inferential testing here; that's the integration Rmd's job)
- `data/derived/coding/reliability_manifest.json`

## What this notebook does NOT do

- No hypothesis testing. No chi-squares. The integration Rmd does all inference.
- No manuscript figures. The Rmd produces plots in ggplot for stylistic consistency.
- No mixing with chat-log codes (Notebook 08 has its own reliability pipeline).


## 1. Environment setup

In [ ]:
import sys, os, json, hashlib
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# Install krippendorff if not present
try:
    import krippendorff
except ImportError:
    print('Installing krippendorff...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'krippendorff'])
    import krippendorff

from scipy import stats as scipy_stats
from sklearn.metrics import cohen_kappa_score

print(f'pandas: {pd.__version__}')
print(f'numpy:  {np.__version__}')
print(f'krippendorff: {krippendorff.__version__ if hasattr(krippendorff, "__version__") else "(unknown)"}')


## 2. Project paths

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
                        'DGK Lab - AI Training Data, CharacterAI, Tort Liability/'
                        'Analysis/Study 1')
else:
    PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == 'NLP') else Path.cwd()

DATA_DERIVED = PROJECT_ROOT / 'data' / 'derived'
CODING_DIR   = DATA_DERIVED / 'coding'
RUBRIC_DIR   = DATA_DERIVED / 'rubric'

PATH_LONG    = CODING_DIR / 'oe_coded_long.parquet'
PATH_RUBRIC  = RUBRIC_DIR / 'rubric_v2.json'

for p in [PATH_LONG, PATH_RUBRIC]:
    if not p.exists():
        raise FileNotFoundError(f'Missing input: {p}')

print(f'Project root: {PROJECT_ROOT}')
print(f'Coding dir:   {CODING_DIR}')


## 3. Load coded data and rubric

In [ ]:
long_df = pd.read_parquet(PATH_LONG)
print(f'Long-form: {long_df.shape[0]:,} rows \u00d7 {long_df.shape[1]} cols')
print(f'  Participants: {long_df["sona_id"].nunique()}')
print(f'  Questions:    {long_df["oe_field"].nunique()}')
print(f'  Codes:        {long_df.groupby("oe_field")["code_id"].nunique().to_dict()}')
print(f'  LLMs:         {long_df["llm_id"].unique().tolist()}')
print(f'  Value dist:   {long_df["value"].value_counts().to_dict()}')

with open(PATH_RUBRIC) as f:
    rubric = json.load(f)
print(f'\nRubric version: {rubric["version"]} (developed by {rubric["rubric_developed_by"]})')


## 4. Pivot to wide-form for reliability computation

For each `(sona_id, oe_field, code_id)` triple, we want three columns: one per LLM, with the binary 0/1 value. NaN where an LLM didn't return a code for that response (Gemini's ~15 missing units).


In [ ]:
LLM_ORDER = ['sonnet-4-6', 'gpt-5-4-mini', 'gemini-3-flash']

wide = long_df.pivot_table(
    index=['sona_id', 'condition_label', 'oe_field', 'code_id', 'category', 'coding_type'],
    columns='llm_id',
    values='value',
    aggfunc='first',
).reset_index()

# Reorder columns
meta_cols = ['sona_id', 'condition_label', 'oe_field', 'code_id', 'category', 'coding_type']
wide = wide[meta_cols + LLM_ORDER]

# Diagnostic
n_complete = wide[LLM_ORDER].notna().all(axis=1).sum()
n_partial  = ((wide[LLM_ORDER].notna().sum(axis=1) == 2)).sum()
print(f'Wide-form: {len(wide):,} rows')
print(f'  All 3 LLMs:  {n_complete:,} ({100*n_complete/len(wide):.1f}%)')
print(f'  2 of 3 LLMs: {n_partial:,} ({100*n_partial/len(wide):.1f}%)')

# Sanity: only Gemini should ever be missing
for llm in LLM_ORDER:
    n_miss = wide[llm].isna().sum()
    print(f'  Missing for {llm}: {n_miss:,}')


## 5. Compute reliability statistics

For each (oe_field, code_id), we compute three statistics across the three LLMs:

- **Cohen's κ (mean pairwise).** Average of three pairwise Cohen's κ values (Sonnet-GPT, Sonnet-Gemini, GPT-Gemini). Standard but pairwise-only.
- **Fleiss's κ.** Omnibus across all three raters. Single number, well-behaved with three coders.
- **Krippendorff's α.** Handles missing data natively (the 15 Gemini gaps). Increasingly the field standard for LLM-coding reliability.

We bootstrap 95% CIs on Krippendorff's α (the most defensible single statistic, used here for CI reporting only — tier assignment still uses all three).


In [ ]:
def fleiss_kappa(ratings_matrix: np.ndarray) -> float:
    """Fleiss's kappa for an N x R matrix where each cell is 0 or 1 (binary).

    ratings_matrix: shape (N items, R raters), values in {0, 1}, NaN allowed.
    Items with any NaN are excluded (Fleiss requires complete cases).
    """
    # Drop rows with any NaN
    complete = ratings_matrix[~np.isnan(ratings_matrix).any(axis=1)]
    if len(complete) < 2:
        return np.nan
    n, r = complete.shape
    # Counts per category (0 and 1) per item
    counts = np.zeros((n, 2))
    counts[:, 1] = complete.sum(axis=1)
    counts[:, 0] = r - counts[:, 1]
    # P_i (proportion of pairs agreeing per item)
    P_i = (counts * (counts - 1)).sum(axis=1) / (r * (r - 1))
    P_bar = P_i.mean()
    # P_e (expected agreement by chance)
    p_j = counts.sum(axis=0) / (n * r)
    P_e = (p_j ** 2).sum()
    if np.isclose(1 - P_e, 0):
        return 1.0  # perfect agreement, no chance variation
    return (P_bar - P_e) / (1 - P_e)


def krippendorff_alpha_binary(ratings_matrix: np.ndarray) -> float:
    """Krippendorff's alpha for binary data, R x N format expected by the package."""
    # krippendorff package wants raters as rows, items as columns
    # NaN is the package's missing-data marker — pass through unchanged
    if ratings_matrix.shape[0] == 0:
        return np.nan
    matrix_T = ratings_matrix.T  # (R, N)
    # Need at least 2 paired observations for alpha to be defined
    if not np.isfinite(matrix_T).any():
        return np.nan
    try:
        alpha = krippendorff.alpha(reliability_data=matrix_T, level_of_measurement='nominal')
        return alpha
    except (ValueError, ZeroDivisionError):
        return np.nan


def pairwise_cohens(ratings_matrix: np.ndarray) -> Tuple[List[float], float]:
    """Returns (list of pairwise Cohen's kappas, mean)."""
    n_raters = ratings_matrix.shape[1]
    kappas = []
    for i in range(n_raters):
        for j in range(i + 1, n_raters):
            a, b = ratings_matrix[:, i], ratings_matrix[:, j]
            mask = ~(np.isnan(a) | np.isnan(b))
            if mask.sum() < 2:
                kappas.append(np.nan)
                continue
            try:
                k = cohen_kappa_score(a[mask], b[mask])
                kappas.append(k)
            except (ValueError, ZeroDivisionError):
                kappas.append(np.nan)
    valid = [k for k in kappas if not np.isnan(k)]
    mean_k = np.mean(valid) if valid else np.nan
    return kappas, mean_k


def pabak(ratings_matrix: np.ndarray) -> float:
    """Prevalence- and bias-adjusted kappa. Robust to extreme base rates.

    For binary 3-rater data, PABAK = 2 * p_observed_agreement - 1
    where p_observed_agreement is the mean pairwise agreement proportion.
    """
    n_raters = ratings_matrix.shape[1]
    agreements = []
    for i in range(n_raters):
        for j in range(i + 1, n_raters):
            a, b = ratings_matrix[:, i], ratings_matrix[:, j]
            mask = ~(np.isnan(a) | np.isnan(b))
            if mask.sum() < 1:
                continue
            agreements.append((a[mask] == b[mask]).mean())
    if not agreements:
        return np.nan
    return 2 * np.mean(agreements) - 1


def bootstrap_alpha_ci(ratings_matrix: np.ndarray, n_boot: int = 1000,
                        seed: int = 42) -> Tuple[float, float]:
    """Bootstrap 95% CI for Krippendorff's alpha by resampling items."""
    n = ratings_matrix.shape[0]
    if n < 10:
        return (np.nan, np.nan)
    rng = np.random.RandomState(seed)
    alphas = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        boot = ratings_matrix[idx]
        a = krippendorff_alpha_binary(boot)
        if not np.isnan(a):
            alphas.append(a)
    if len(alphas) < 50:
        return (np.nan, np.nan)
    return (np.percentile(alphas, 2.5), np.percentile(alphas, 97.5))


print('Reliability functions defined.')


## 6. Compute reliability table for every (oe_field × code_id)

In [ ]:
from tqdm.auto import tqdm

rows = []
for (oe_field, code_id), grp in tqdm(wide.groupby(['oe_field', 'code_id']),
                                       desc='Computing reliability'):
    ratings = grp[LLM_ORDER].values.astype(float)  # N x 3

    # Sample size: rows with at least 2 non-missing
    n_complete = (~np.isnan(ratings)).all(axis=1).sum()
    n_partial  = ((~np.isnan(ratings)).sum(axis=1) == 2).sum()
    n_total    = (~np.isnan(ratings)).any(axis=1).sum()

    # Prevalence: mean across all non-missing cells, then across LLMs
    prev_per_llm = np.nanmean(ratings, axis=0)
    prevalence = np.nanmean(prev_per_llm)

    # Three reliability statistics
    pair_ks, mean_cohen = pairwise_cohens(ratings)
    fleiss = fleiss_kappa(ratings)
    alpha  = krippendorff_alpha_binary(ratings)

    # PABAK (only meaningful when prevalence is extreme)
    pabak_val = pabak(ratings) if (prevalence < 0.10 or prevalence > 0.90) else np.nan

    # Bootstrap CI on alpha (only if we have enough rows)
    ci_lo, ci_hi = bootstrap_alpha_ci(ratings, n_boot=1000)

    # Conjunctive tier assignment: all three statistics must clear threshold
    stats_for_tier = [mean_cohen, fleiss, alpha]
    valid_stats = [s for s in stats_for_tier if not np.isnan(s)]
    if not valid_stats:
        tier = 'unreliable'
    elif all(s >= 0.60 for s in valid_stats):
        tier = 'confirmatory'
    elif all(s >= 0.40 for s in valid_stats):
        tier = 'exploratory'
    else:
        tier = 'descriptive_only'

    rows.append({
        'oe_field':           oe_field,
        'code_id':            code_id,
        'category':           grp['category'].iloc[0],
        'coding_type':        grp['coding_type'].iloc[0],
        'n_total':            int(n_total),
        'n_complete_3_llms':  int(n_complete),
        'n_partial_2_llms':   int(n_partial),
        'prevalence':         round(prevalence, 4),
        'prev_sonnet':        round(prev_per_llm[0], 4),
        'prev_gpt':           round(prev_per_llm[1], 4),
        'prev_gemini':        round(prev_per_llm[2], 4),
        'cohen_sonnet_gpt':    round(pair_ks[0], 4) if not np.isnan(pair_ks[0]) else None,
        'cohen_sonnet_gemini': round(pair_ks[1], 4) if not np.isnan(pair_ks[1]) else None,
        'cohen_gpt_gemini':    round(pair_ks[2], 4) if not np.isnan(pair_ks[2]) else None,
        'cohen_mean':          round(mean_cohen, 4) if not np.isnan(mean_cohen) else None,
        'fleiss_kappa':        round(fleiss, 4) if not np.isnan(fleiss) else None,
        'krippendorff_alpha':  round(alpha, 4) if not np.isnan(alpha) else None,
        'alpha_ci_lo_95':      round(ci_lo, 4) if not np.isnan(ci_lo) else None,
        'alpha_ci_hi_95':      round(ci_hi, 4) if not np.isnan(ci_hi) else None,
        'pabak':               round(pabak_val, 4) if not np.isnan(pabak_val) else None,
        'tier':                tier,
    })

reliability_df = pd.DataFrame(rows)
print(f'\nReliability table: {len(reliability_df)} rows (oe_field \u00d7 code_id combinations)')
print(f'\nTier distribution overall:')
print(reliability_df['tier'].value_counts())
print(f'\nTier distribution per question:')
print(reliability_df.groupby(['oe_field', 'tier']).size().unstack(fill_value=0))


## 7. Reliability summary

Print the full reliability table sorted by question, then by tier (confirmatory first).


In [ ]:
tier_order = {'confirmatory': 0, 'exploratory': 1, 'descriptive_only': 2, 'unreliable': 3}
reliability_df['_tier_sort'] = reliability_df['tier'].map(tier_order)

display_cols = ['oe_field', 'category', 'code_id', 'prevalence', 'n_total',
                'cohen_mean', 'fleiss_kappa', 'krippendorff_alpha', 'tier']

for q in ['oe_experience', 'oe_person_or_tool', 'oe_responsibility', 'oe_compensation']:
    sub = reliability_df[reliability_df['oe_field'] == q].sort_values(['_tier_sort', 'category', 'code_id'])
    print(f'\n=== {q} ===')
    print(sub[display_cols].to_string(index=False))

reliability_df = reliability_df.drop(columns=['_tier_sort'])


## 8. Disagreement diagnostics

For codes that fall below the confirmatory threshold, where does the disagreement come from? Identify the "odd-one-out" — the LLM most often disagreeing with the consensus of the other two — for each code.


In [ ]:
def odd_one_out_analysis(wide_df, llm_cols=LLM_ORDER):
    """For each (oe_field, code_id), tally how often each LLM is the lone dissenter
    (i.e., disagrees with the other two who agree with each other)."""
    rows = []
    for (oe_field, code_id), grp in wide_df.groupby(['oe_field', 'code_id']):
        ratings = grp[llm_cols].values.astype(float)
        # Only rows where all 3 are present
        complete = ratings[~np.isnan(ratings).any(axis=1)]
        if len(complete) == 0:
            continue
        # Count: for each row, is exactly one LLM disagreeing?
        unanimous = (complete.sum(axis=1).isin([0, 3]) if hasattr(complete.sum(axis=1), 'isin')
                     else np.isin(complete.sum(axis=1), [0, 3]))
        # When sum == 1: one rater said 1, two said 0 — the "1" rater is the odd one
        # When sum == 2: one rater said 0, two said 1 — the "0" rater is the odd one
        odd_counts = {llm: 0 for llm in llm_cols}
        for row in complete:
            s = row.sum()
            if s == 1:
                idx = int(np.argmax(row == 1))
                odd_counts[llm_cols[idx]] += 1
            elif s == 2:
                idx = int(np.argmax(row == 0))
                odd_counts[llm_cols[idx]] += 1
        total_disagreement = sum(odd_counts.values())
        rows.append({
            'oe_field': oe_field,
            'code_id':  code_id,
            'n_complete': len(complete),
            'n_unanimous': int(unanimous.sum()),
            'n_disagreement': total_disagreement,
            'pct_unanimous': round(unanimous.sum() / len(complete), 3) if len(complete) > 0 else np.nan,
            'odd_sonnet':   odd_counts['sonnet-4-6'],
            'odd_gpt':      odd_counts['gpt-5-4-mini'],
            'odd_gemini':   odd_counts['gemini-3-flash'],
        })
    return pd.DataFrame(rows)

odd_df = odd_one_out_analysis(wide)

# Merge with tier so we can focus on non-confirmatory codes
odd_with_tier = odd_df.merge(
    reliability_df[['oe_field', 'code_id', 'tier']], on=['oe_field', 'code_id']
)

# Show codes where one LLM is markedly the most frequent dissenter
print('Codes where one LLM is the lone dissenter more often than the others:')
print('(only shown for codes with at least 10 total disagreements)')
problem_codes = odd_with_tier[odd_with_tier['n_disagreement'] >= 10].copy()
problem_codes['max_odd_pct'] = problem_codes[['odd_sonnet','odd_gpt','odd_gemini']].max(axis=1) / problem_codes['n_disagreement']
problem_codes['main_dissenter'] = problem_codes[['odd_sonnet','odd_gpt','odd_gemini']].idxmax(axis=1)
print(problem_codes.sort_values('max_odd_pct', ascending=False).head(15).to_string(index=False))


## 9. Per-condition reliability check

Is reliability equivalent across HighA and LowA conditions? If LLMs agree more on one condition's responses than the other's, that's a confound to flag in Methods.


In [ ]:
rows = []
for (oe_field, code_id), grp in wide.groupby(['oe_field', 'code_id']):
    for cond in ['HighA', 'LowA']:
        sub = grp[grp['condition_label'] == cond]
        if len(sub) < 5:
            continue
        ratings = sub[LLM_ORDER].values.astype(float)
        alpha = krippendorff_alpha_binary(ratings)
        prev  = np.nanmean(ratings)
        rows.append({
            'oe_field':            oe_field,
            'code_id':             code_id,
            'condition':           cond,
            'n':                   len(sub),
            'prevalence':          round(prev, 4),
            'krippendorff_alpha':  round(alpha, 4) if not np.isnan(alpha) else None,
        })
per_condition_df = pd.DataFrame(rows)

# Pivot to side-by-side
per_cond_wide = per_condition_df.pivot_table(
    index=['oe_field', 'code_id'], columns='condition',
    values='krippendorff_alpha', aggfunc='first'
).reset_index()
per_cond_wide['alpha_diff_HighA_minus_LowA'] = (
    per_cond_wide['HighA'] - per_cond_wide['LowA']
).round(4)

print('Per-condition Krippendorff\'s α (selected codes; full table written to disk):')
print(per_cond_wide.sort_values('alpha_diff_HighA_minus_LowA', key=abs, ascending=False).head(15).to_string(index=False))

# Summary diagnostic: is one condition systematically more reliable?
mean_diff = per_cond_wide['alpha_diff_HighA_minus_LowA'].mean()
sd_diff   = per_cond_wide['alpha_diff_HighA_minus_LowA'].std()
n_extreme = (per_cond_wide['alpha_diff_HighA_minus_LowA'].abs() > 0.20).sum()
print(f'\nSummary across all codes:')
print(f'  Mean(α-HighA − α-LowA) = {mean_diff:+.4f}  (positive = HighA more reliable)')
print(f'  SD of differences      = {sd_diff:.4f}')
print(f'  Codes with |diff| > .20 = {n_extreme} of {len(per_cond_wide)}')


## 10. Build consensus dataset

For each `(sona_id, oe_field, code_id)`, apply the majority-vote rule:

- All 3 LLMs present: code value = 1 iff ≥ 2 agree on 1
- Only 2 of 3 present: code value = 1 iff both agree on 1; **ties (1 yes, 1 no) → 0** (conservative)
- Only 0 or 1 LLM present: drop (shouldn't occur given Notebook 04 outputs)

Confidence weighting is not used here. Confidence is preserved in the long-form outputs for anyone who wants to compute it as a robustness check.


In [ ]:
def majority_vote_conservative(row):
    """Apply majority-vote rule with conservative tie-break.

    Returns (consensus_value, n_coders_used, agreement_strength).
    """
    vals = [row[llm] for llm in LLM_ORDER]
    non_nan = [v for v in vals if not pd.isna(v)]
    n = len(non_nan)
    if n == 0:
        return (np.nan, 0, np.nan)
    s = sum(non_nan)
    if n == 3:
        return (1 if s >= 2 else 0, 3, s / 3.0)
    elif n == 2:
        if s == 2:
            return (1, 2, 1.0)
        elif s == 0:
            return (0, 2, 1.0)
        else:
            return (0, 2, 0.5)  # tied, conservative-zero
    else:
        return (non_nan[0], 1, np.nan)

# Compute per-row consensus
results = wide.apply(lambda r: pd.Series(majority_vote_conservative(r),
                                          index=['consensus', 'n_coders', 'agreement_strength']),
                     axis=1)
consensus_long = pd.concat([wide[['sona_id', 'condition_label', 'oe_field', 'code_id',
                                    'category', 'coding_type']], results], axis=1)
consensus_long['consensus'] = consensus_long['consensus'].astype('Int64')

print(f'Consensus long-form: {len(consensus_long)} rows')
print(f'  Consensus value distribution: {consensus_long["consensus"].value_counts().to_dict()}')
print(f'  Coder coverage: {consensus_long["n_coders"].value_counts().to_dict()}')

# Build per-question wide-form parquets. One row per sona_id, columns for that
# question's codes only. This is what the integration Rmd will read.
consensus_wide_per_question = {}
for q in consensus_long['oe_field'].unique():
    sub = consensus_long[consensus_long['oe_field'] == q]
    wq = sub.pivot_table(
        index=['sona_id', 'condition_label'],
        columns='code_id',
        values='consensus',
        aggfunc='first',
    ).reset_index()
    # Convert to plain int (NaN \u2192 0 only for codes whose underlying value was
    # genuinely missing because no LLM coded; otherwise consensus is already 0/1).
    # The pivot fills with NaN only when (sona, code) is absent from the table,
    # which shouldn't happen — every (sona, code) was attempted in Notebook 04.
    code_cols = [c for c in wq.columns if c not in ('sona_id', 'condition_label')]
    print(f'  {q}: {len(wq)} rows \u00d7 {len(code_cols)} codes')
    consensus_wide_per_question[q] = wq

# Also build a combined long-form-but-wide-by-question for cases where the Rmd wants everything
# (used for cross-question analyses if any)


## 11. Descriptive prevalence by condition

For each code, what's the consensus-value prevalence in HighA vs LowA? No inferential testing here — this is descriptive table fodder for the integration Rmd to test.


In [ ]:
prev_by_cond = consensus_long.groupby(['oe_field', 'category', 'code_id', 'condition_label']).agg(
    n=('consensus', 'size'),
    n_present=('consensus', lambda s: s.fillna(0).sum()),
    prevalence=('consensus', lambda s: s.fillna(0).mean()),
).reset_index()

prev_wide = prev_by_cond.pivot_table(
    index=['oe_field', 'category', 'code_id'],
    columns='condition_label',
    values='prevalence',
).reset_index()
prev_wide['HighA_minus_LowA'] = (prev_wide['HighA'] - prev_wide['LowA']).round(4)
prev_wide['HighA'] = prev_wide['HighA'].round(4)
prev_wide['LowA']  = prev_wide['LowA'].round(4)

# Merge with tier so the Rmd can filter to confirmatory codes
prev_wide = prev_wide.merge(
    reliability_df[['oe_field', 'code_id', 'tier', 'prevalence']],
    on=['oe_field', 'code_id'],
    suffixes=('', '_overall'),
)

print('Codes with largest HighA-LowA difference (top 12, all tiers):')
print(prev_wide.reindex(prev_wide['HighA_minus_LowA'].abs().sort_values(ascending=False).index)
      .head(12)[['oe_field', 'category', 'code_id', 'HighA', 'LowA', 'HighA_minus_LowA', 'tier']]
      .to_string(index=False))


## 12. Sanity checks before writing outputs

In [ ]:
checks = []

# (1) No code has 100% or 0% prevalence overall
extreme_high = (reliability_df['prevalence'] > 0.99).sum()
extreme_low  = (reliability_df['prevalence'] < 0.005).sum()
checks.append(('No codes at 100% prevalence', extreme_high == 0, f'{extreme_high} codes >99%'))
checks.append(('No codes at <0.5% prevalence', extreme_low == 0, f'{extreme_low} codes <0.5%'))

# (2) All response-units have at least 2 coders
underconfided = (consensus_long['n_coders'] < 2).sum()
checks.append(('All consensus rows have \u22652 coders', underconfided == 0, f'{underconfided} rows with <2 coders'))

# (3) Per-question wide-forms preserve their sona_id counts.
# Note: per-question sona_id counts will differ across questions because
# some participants wrote substantive responses on some questions but not
# others (Notebook 04 filtered to >=3-word responses). What we verify here
# is that for each question, the wide-form preserves the same sona_id set
# that's present in the long-form for that question.
all_match = True
mismatch_msgs = []
for q, wq in consensus_wide_per_question.items():
    long_sona = set(consensus_long[consensus_long['oe_field'] == q]['sona_id'].unique())
    wide_sona = set(wq['sona_id'].unique())
    if long_sona != wide_sona:
        all_match = False
        mismatch_msgs.append(f'{q}: long={len(long_sona)} wide={len(wide_sona)}')
checks.append(('Per-question wide-forms preserve sona_id sets', all_match,
               '; '.join(mismatch_msgs) if mismatch_msgs else 'all match'))

# (4) Tier distribution is non-degenerate (we should have a mix)
tier_counts = reliability_df['tier'].value_counts()
n_tiers = (tier_counts > 0).sum()
checks.append(('Tier distribution non-degenerate', n_tiers >= 2,
               f'tiers present: {tier_counts.to_dict()}'))

# (5) Per-LLM prevalence is plausible (no LLM at >90% or <1% overall)
for llm_short, full in [('sonnet','prev_sonnet'), ('gpt','prev_gpt'), ('gemini','prev_gemini')]:
    mean_prev = reliability_df[full].mean()
    sane = 0.05 < mean_prev < 0.5
    checks.append((f'{llm_short} mean prevalence in [.05, .50]', sane,
                   f'mean={mean_prev:.3f}'))

# (6) Each question has roughly the participant counts we'd expect
# (at least 100 of the 130 analytic participants should have answered each OE)
for q in consensus_long['oe_field'].unique():
    n_q = consensus_long[consensus_long['oe_field'] == q]['sona_id'].nunique()
    sane = n_q >= 100
    checks.append((f'{q} has \u2265100 substantive respondents', sane, f'{n_q} participants'))

# Print
print('-' * 70)
all_pass = True
for name, passed, detail in checks:
    marker = '[OK]' if passed else '[FAIL]'
    if not passed:
        all_pass = False
    print(f'  {marker:<8s} {name}')
    if not passed:
        print(f'           -> {detail}')
print('-' * 70)
print('All checks passed.' if all_pass else 'SANITY CHECKS FAILED. Review above.')


## 13. Write outputs

In [ ]:
# Reliability table
rel_path = CODING_DIR / 'reliability_table.csv'
reliability_df.to_csv(rel_path, index=False)

# Per-condition reliability
per_cond_path = CODING_DIR / 'reliability_per_condition.csv'
per_cond_wide.to_csv(per_cond_path, index=False)

# Consensus long (master output, one row per sona \u00d7 oe \u00d7 code)
cons_long_path = CODING_DIR / 'consensus_long.parquet'
consensus_long.to_parquet(cons_long_path, index=False)

# Per-question wide-form parquets (one file per OE; what the Rmd reads)
per_q_paths = {}
for q, wq in consensus_wide_per_question.items():
    p = CODING_DIR / f'consensus_wide_{q}.parquet'
    wq.to_parquet(p, index=False)
    per_q_paths[q] = p

# Code prevalence by condition (descriptive only, for Rmd to test)
prev_path = CODING_DIR / 'code_prevalence_by_condition.csv'
prev_wide.to_csv(prev_path, index=False)

# Manifest
manifest = {
    'generated_at': datetime.now().isoformat(timespec='seconds'),
    'rubric_version': rubric['version'],
    'n_participants_coded': int(consensus_long['sona_id'].nunique()),
    'n_questions': int(consensus_long['oe_field'].nunique()),
    'n_codes': int(reliability_df.groupby('oe_field')['code_id'].nunique().sum()),
    'tier_assignment_rule': 'conjunctive: cohen_mean, fleiss_kappa, krippendorff_alpha ALL >= threshold',
    'thresholds': {'confirmatory': 0.60, 'exploratory': 0.40},
    'tie_break_rule': 'majority vote across 3 LLMs; 2-of-2-with-1-missing tied -> 0 (conservative)',
    'tier_counts': reliability_df['tier'].value_counts().to_dict(),
    'tier_counts_per_question': (reliability_df.groupby(['oe_field', 'tier'])
                                  .size().unstack(fill_value=0).to_dict()),
    'mean_alpha_by_tier': (reliability_df.groupby('tier')['krippendorff_alpha']
                            .mean().round(3).to_dict()),
    'per_condition_alpha_diff': {
        'mean': round(per_cond_wide['alpha_diff_HighA_minus_LowA'].mean(), 4),
        'sd':   round(per_cond_wide['alpha_diff_HighA_minus_LowA'].std(), 4),
        'n_extreme_|diff|>0.20': int((per_cond_wide['alpha_diff_HighA_minus_LowA'].abs() > 0.20).sum()),
    },
    'outputs': {
        'reliability_table':           str(rel_path.relative_to(PROJECT_ROOT)),
        'consensus_long':              str(cons_long_path.relative_to(PROJECT_ROOT)),
        'consensus_wide_per_question': {q: str(p.relative_to(PROJECT_ROOT)) for q, p in per_q_paths.items()},
        'reliability_per_condition':   str(per_cond_path.relative_to(PROJECT_ROOT)),
        'code_prevalence_by_condition': str(prev_path.relative_to(PROJECT_ROOT)),
    },
}
manifest_path = CODING_DIR / 'reliability_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Wrote:')
print(f'  {rel_path}')
print(f'  {per_cond_path}')
print(f'  {cons_long_path}')
for q, p in per_q_paths.items():
    print(f'  {p}')
print(f'  {prev_path}')
print(f'  {manifest_path}')


## 14. Wrap-up

In [ ]:
print('=' * 70)
print('NOTEBOOK 05 COMPLETE')
print('=' * 70)
print(f'Codes assessed:       {len(reliability_df)}')
print(f'  confirmatory:       {(reliability_df["tier"] == "confirmatory").sum()}')
print(f'  exploratory:        {(reliability_df["tier"] == "exploratory").sum()}')
print(f'  descriptive only:   {(reliability_df["tier"] == "descriptive_only").sum()}')
print(f'  unreliable:         {(reliability_df["tier"] == "unreliable").sum()}')
print()
print(f'Reliability statistics (median across all codes):')
print(f"  Cohen's \u03ba (mean pairwise): {reliability_df['cohen_mean'].median():.3f}")
print(f"  Fleiss's \u03ba:                {reliability_df['fleiss_kappa'].median():.3f}")
print(f"  Krippendorff's \u03b1:           {reliability_df['krippendorff_alpha'].median():.3f}")
print()
print(f'Next: Notebook 06 (topic modeling).')
print(f'      Integration into manuscript happens in study1_nlp_integration.Rmd.')
